# 1. Load Raw +Tour Dataset

- Start PySpark on Windows.
- Load files from `Datasets/Raw`.
- Show schema, sample rows, row count, and columns.


## Configure Python path for PySpark on Windows

In [1]:
import os
import sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Python executable:")
print(sys.executable)

Python executable:
/usr/bin/python3


## Start Spark

In [2]:
import os
import sys
from pyspark.sql import *


def create_spark_session(app_name="PlusTourPySpark"):

    spark = (
        SparkSession.builder
        .appName(app_name)
        .master("local[*]")
        .config("spark.hadoop.fs.defaultFS", "file:///")
        .config(
        "spark.jars.repositories",
        "https://repos.spark-packages.org/"
        )
        .config(
            "spark.jars.packages",
            "io.graphframes:graphframes-spark4_2.13:0.9.3"
        )
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    return spark

## Set project root and raw dataset path

In [3]:
import sys
import importlib
from pathlib import Path

current_path = Path.cwd()

if current_path.name.lower() == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

importlib.invalidate_caches()

spark = create_spark_session()

print("Project root:", project_root)
print("Python executable:", sys.executable)
print("Spark version:", spark.version)
print("Spark started successfully.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/02 16:02:21 WARN Utils: Your hostname, duongthai-VirtualBox, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/06/02 16:02:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
https://repos.spark-packages.org/ added as a remote repository with the name: repo-1
:: loading settings :: url = jar:file:/home/duongthai/spark-4.1.1-bin-hadoop3/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/duongthai/.ivy2.5.2/cache
The jars for the packages stored in: /home/duongthai/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5be69b89-c5f7-4b08-b441-c938a0ac66f7;1.0
	confs: [default]
	found io.graphframes#graphframes-spark4_2.13;0.9.3 in central
:: resolution report :: resolve 2215ms :: artifacts dl 46ms
	:: modules 

Project root: /home/duongthai/Downloads/DoAnBigData/smart-tourism
Python executable: /usr/bin/python3
Spark version: 4.1.1
Spark started successfully.


## Defining project and dataset path

The variable `raw_data_path` is then used by later cells to locate and load files from `Datasets/Raw`.

In [4]:
from pathlib import Path

current_path = Path.cwd()

if current_path.name.lower() == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

raw_data_path = project_root / "Datasets" / "Raw"

print("Project root:", project_root)
print("Raw data path:", raw_data_path)

Project root: /home/duongthai/Downloads/DoAnBigData/smart-tourism
Raw data path: /home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw


## List raw dataset files

In [5]:
if raw_data_path.exists():
    all_items = list(raw_data_path.rglob("*"))
    files = [p for p in all_items if p.is_file()]

    print("Total files found:", len(files))
    print("First 30 files:")
    for file in files[:30]:
        print(file.relative_to(project_root))
else:
    raise FileNotFoundError(f"Cannot find raw data folder: {raw_data_path}")

Total files found: 39
First 30 files:
Datasets/Raw/Barcelona/distanceMatrix.json
Datasets/Raw/Barcelona/touristsVisits.csv
Datasets/Raw/Barcelona/POIs.csv
Datasets/Raw/Toronto/distanceMatrix.json
Datasets/Raw/Toronto/touristsVisits.csv
Datasets/Raw/Toronto/POIs.csv
Datasets/Raw/Osaka/distanceMatrix.json
Datasets/Raw/Osaka/touristsVisits.csv
Datasets/Raw/Osaka/POIs.csv
Datasets/Raw/Perth/distanceMatrix.json
Datasets/Raw/Perth/touristsVisits.csv
Datasets/Raw/Perth/POIs.csv
Datasets/Raw/Glasgow/distanceMatrix.json
Datasets/Raw/Glasgow/touristsVisits.csv
Datasets/Raw/Glasgow/POIs.csv
Datasets/Raw/Edinburgh/distanceMatrix.json
Datasets/Raw/Edinburgh/touristsVisits.csv
Datasets/Raw/Edinburgh/POIs.csv
Datasets/Raw/Athens/distanceMatrix.json
Datasets/Raw/Athens/touristsVisits.csv
Datasets/Raw/Athens/POIs.csv
Datasets/Raw/NewDelhi/distanceMatrix.json
Datasets/Raw/NewDelhi/touristsVisits.csv
Datasets/Raw/NewDelhi/POIs.csv
Datasets/Raw/Budapest/distanceMatrix.json
Datasets/Raw/Budapest/touristsVi

## Load raw dataset

This reads all files under `Datasets/Raw`, including nested folders.

In [6]:
import pandas as pd
import json

from pathlib import Path
from pyspark.sql.functions import col

csv_files = list(raw_data_path.rglob("*.csv"))
json_files = list(raw_data_path.rglob("*.json"))

print("CSV files found:", len(csv_files))
print("JSON files found:", len(json_files))

for f in csv_files[:10]:
    print(f)

for f in json_files[:10]:
    print(f)


CSV files found: 26
JSON files found: 13
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Barcelona/touristsVisits.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Barcelona/POIs.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Toronto/touristsVisits.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Toronto/POIs.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Osaka/touristsVisits.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Osaka/POIs.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Perth/touristsVisits.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Perth/POIs.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Glasgow/touristsVisits.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Glasgow/POIs.csv
/home/duongthai/Downloads/DoAnBigData/smart-tourism/Datasets/Raw/Barcelona/distanceMatrix.jso

In [7]:
# Load touristsVisits.csv

visit_list = []

for file in csv_files:
    if file.name == "touristsVisits.csv":
        temp_pdf = pd.read_csv(f"file://{file}")
        temp_pdf["city"] = file.parent.name
        visit_list.append(temp_pdf)

visit_pdf = pd.concat(visit_list, ignore_index=True)
visit_df = spark.createDataFrame(visit_pdf)

print("Visits dataset loaded.")
visit_df.printSchema()
visit_df.show(10, truncate=False)


Visits dataset loaded.
root
 |-- photoID: long (nullable = true)
 |-- userID: string (nullable = true)
 |-- dateTaken: long (nullable = true)
 |-- poiID: long (nullable = true)
 |-- poiTheme: string (nullable = true)
 |-- seqID: long (nullable = true)
 |-- city: string (nullable = true)



26/06/02 16:03:26 WARN TaskSetManager: Stage 0 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):                                  (0 + 1) / 1]
  File "/home/duongthai/spark-4.1.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/home/duongthai/spark-4.1.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


+-------+-------------+----------+-----+--------+-----+---------+
|photoID|userID       |dateTaken |poiID|poiTheme|seqID|city     |
+-------+-------------+----------+-----+--------+-----+---------+
|1      |100108211@N06|1501791155|29   |Museum  |1    |Barcelona|
|2      |100108211@N06|1501792349|29   |Museum  |1    |Barcelona|
|3      |100108211@N06|1501792579|29   |Museum  |1    |Barcelona|
|4      |100108211@N06|1501793533|29   |Museum  |1    |Barcelona|
|5      |100108211@N06|1501794789|29   |Museum  |1    |Barcelona|
|6      |100108211@N06|1501803723|29   |Museum  |1    |Barcelona|
|7      |100108211@N06|1501803727|29   |Museum  |1    |Barcelona|
|8      |100108211@N06|1501803955|29   |Museum  |1    |Barcelona|
|9      |100108211@N06|1501804008|29   |Museum  |1    |Barcelona|
|10     |100108211@N06|1501804091|29   |Museum  |1    |Barcelona|
+-------+-------------+----------+-----+--------+-----+---------+
only showing top 10 rows


In [8]:
# Load POIs.csv

poi_list = []

for file in csv_files:
    if file.name == "POIs.csv":
        temp_pdf = pd.read_csv(f"file://{file}")
        temp_pdf["city"] = file.parent.name
        poi_list.append(temp_pdf)

poi_pdf = pd.concat(poi_list, ignore_index=True)
poi_df = spark.createDataFrame(poi_pdf)

print("POI dataset loaded.")
poi_df.printSchema()
poi_df.show(10, truncate=False)

POI dataset loaded.
root
 |-- poiID: long (nullable = true)
 |-- poiName: string (nullable = true)
 |-- poiLat: double (nullable = true)
 |-- poiLon: double (nullable = true)
 |-- poiTheme: string (nullable = true)
 |-- city: string (nullable = true)



+-----+-------------------+----------+---------+---------+---------+
|poiID|poiName            |poiLat    |poiLon   |poiTheme |city     |
+-----+-------------------+----------+---------+---------+---------+
|1    |Camp_Nou           |41.380896 |2.1228198|Sport    |Barcelona|
|2    |Park_Guell         |41.4144948|2.1526945|Park     |Barcelona|
|3    |Placa_de_Catalunya |41.3870154|2.1700471|Shopping |Barcelona|
|4    |Casa_Batllo        |41.3916384|2.1647698|Education|Barcelona|
|5    |La_Maquinista      |41.4425713|2.200075 |Shopping |Barcelona|
|6    |Diagonal_Mar       |41.4099658|2.2166369|Shopping |Barcelona|
|7    |Maremagnum         |41.375182 |2.182867 |Shopping |Barcelona|
|8    |Aquarium_Barcelona |41.376837 |2.184338 |Zoo      |Barcelona|
|9    |L'illa_Diagonal    |41.3898016|2.1355821|Shopping |Barcelona|
|10   |Arenas_de_Barcelona|41.3763184|2.1493331|Shopping |Barcelona|
+-----+-------------------+----------+---------+---------+---------+
only showing top 10 rows


In [9]:
# load distanceMatrix.json

distance_list = []

for file in json_files:

    if "distance" in file.name.lower():

        with open(file, "r", encoding="utf-8") as f:
            data = pd.read_json(f"file://{file}")

        temp_pdf = pd.DataFrame(data)
        temp_pdf["city"] = file.parent.name
        distance_list.append(temp_pdf)

distance_pdf = pd.concat(distance_list, ignore_index=True)
distance_df = spark.createDataFrame(distance_pdf)

print("Distance Matrix dataset loaded.")
distance_df.printSchema()

distance_df.show(10, truncate=False)

Distance Matrix dataset loaded.
root
 |-- fromPOIid: long (nullable = true)
 |-- fromPOIName: string (nullable = true)
 |-- toPOIid: long (nullable = true)
 |-- toPOIName: string (nullable = true)
 |-- duration: long (nullable = true)
 |-- distance: long (nullable = true)
 |-- city: string (nullable = true)

+---------+-----------+-------+------------------------------------+--------+--------+---------+
|fromPOIid|fromPOIName|toPOIid|toPOIName                           |duration|distance|city     |
+---------+-----------+-------+------------------------------------+--------+--------+---------+
|1        |Camp_Nou   |2      |Park_Guell                          |4578    |5569    |Barcelona|
|1        |Camp_Nou   |3      |Placa_de_Catalunya                  |3759    |5142    |Barcelona|
|1        |Camp_Nou   |4      |Casa_Batllo                         |3328    |4528    |Barcelona|
|1        |Camp_Nou   |6      |Diagonal_Mar                        |6984    |9382    |Barcelona|
|1        |

Traceback (most recent call last):                                  (0 + 1) / 1]
  File "/home/duongthai/spark-4.1.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/home/duongthai/spark-4.1.1-bin-hadoop3/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


## GraphFrames POI Graph
- Build POI Graph
- Calculate PageRank and compare to POI popularity `pop(v_i)`
- Calculate user interest `int_u(c)`
- Find best itinerary using BFS and ShortestPaths

Reference: Section 3 (System Model)

## GraphFrames

- **Vertices** : poiName, poiTheme, lat, lon, city  (more than one POI in one city)
- **Edges**    : each pair of (fromPOI → toPOI) is kept for 2 weights (duration, distance)

Most travel distance matrices represent bidirectional movement between POIs. Therefore, reverse edges are added to make the graph more suitable for tourism route analysis.

In [10]:
from graphframes import GraphFrame
from pyspark.sql import functions as F

vertices = (
    poi_df
    .withColumn("id", F.concat_ws("_", col("city"), col("poiID").cast("string")))
    .select(
        "id",
        col("poiID"),
        col("poiName"),
        col("poiTheme").alias("category"),
        col("poiLat").alias("lat"),
        col("poiLon").alias("lon"),
        col("city"),
    )
)

edges = (
    distance_df
    .withColumn("src", F.concat_ws("_", col("city"), col("fromPOIid").cast("string")))
    .withColumn("dst", F.concat_ws("_", col("city"), col("toPOIid").cast("string")))
    .select("src", "dst",
            col("duration"),   # s
            col("distance"),   # m
            col("city"),
    )

)


reverse_edges = edges.select(
    col("dst").alias("src"),
    col("src").alias("dst"),
    "duration", "distance", "city"
)
edges_full = edges.union(reverse_edges).distinct() 

g = GraphFrame(vertices, edges_full)

print(f"Vertices: {g.vertices.count():,}")
print(f"Edges   : {g.edges.count():,}")
g.vertices.show(5, truncate=False)
g.edges.show(5, truncate=False)

/home/duongthai/.local/lib/python3.10/site-packages/pyspark/sql/classic/dataframe.py:146: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


Vertices: 444


Edges   : 30,486
+-----------+-----+------------------+---------+----------+---------+---------+
|id         |poiID|poiName           |category |lat       |lon      |city     |
+-----------+-----+------------------+---------+----------+---------+---------+
|Barcelona_1|1    |Camp_Nou          |Sport    |41.380896 |2.1228198|Barcelona|
|Barcelona_2|2    |Park_Guell        |Park     |41.4144948|2.1526945|Barcelona|
|Barcelona_3|3    |Placa_de_Catalunya|Shopping |41.3870154|2.1700471|Barcelona|
|Barcelona_4|4    |Casa_Batllo       |Education|41.3916384|2.1647698|Barcelona|
|Barcelona_5|5    |La_Maquinista     |Shopping |41.4425713|2.200075 |Barcelona|
+-----------+-----+------------------+---------+----------+---------+---------+
only showing top 5 rows


+------------+------------+--------+--------+---------+
|src         |dst         |duration|distance|city     |
+------------+------------+--------+--------+---------+
|Barcelona_6 |Barcelona_1 |7189    |9377    |Barcelona|
|Barcelona_23|Barcelona_27|3871    |4664    |Barcelona|
|Toronto_7   |Toronto_24  |1056    |1368    |Toronto  |
|Toronto_18  |Toronto_11  |5766    |7474    |Toronto  |
|Toronto_20  |Toronto_13  |5702    |7386    |Toronto  |
+------------+------------+--------+--------+---------+
only showing top 5 rows


## PageRank for each POI network

In [11]:
pr_results = g.pageRank(resetProbability=0.15, tol=0.01)

pagerank_df = (
    pr_results.vertices
    .select("id", "city", "poiName", "category", "lat", "lon","pagerank")
    .orderBy(col("pagerank").desc())
)

w_pr = Window.partitionBy("city").orderBy(F.desc("pagerank"))

pagerank_df = (
    pagerank_df
    .withColumn("pagerank_rank", F.row_number().over(w_pr))
)

/home/duongthai/.local/lib/python3.10/site-packages/pyspark/sql/classic/dataframe.py:128: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


In [12]:
print('Top 1 PageRank in each city:')
pagerank_df.select("city", "id","poiName", "category", "lat", "lon","pagerank").where(col('pagerank_rank') == 1).show(truncate=False)

Top 1 PageRank in each city:


+---------+-----------+-------------------------------+--------------+----------+----------+------------------+
|city     |id         |poiName                        |category      |lat       |lon       |pagerank          |
+---------+-----------+-------------------------------+--------------+----------+----------+------------------+
|Athens   |Athens_1   |Acropolis_of_Athens            |Historical    |37.9715323|23.7257492|1.0904063221273281|
|Barcelona|Barcelona_1|Camp_Nou                       |Sport         |41.380896 |2.1228198 |1.0898661458805812|
|Budapest |Budapest_1 |S%C3%A1ndor_Palace             |Historical    |47.4978   |19.0378   |1.0968541638456826|
|Edinburgh|Edinburgh_1|Edinburgh_Castle               |Historical    |55.948611 |-3.200833 |1.0887272645879544|
|Glasgow  |Glasgow_1  |Glasgow_Central_railway_station|Transport     |55.858    |-4.258    |1.0902711829368805|
|London   |London_1   |Coca-Cola_London_Eye           |Entertainment |51.503324 |-0.119543 |1.0887272645

## Calculate POI popularity

POI popularity is calculated from tourist visit records.
This metric represents actual tourist behavior in the dataset.

In [13]:
poi_popularity = (
    visit_df
    .groupBy("city", "poiID")
    .count()
    .withColumnRenamed("count", "actual_visits")
    .withColumn("poi_key", F.concat_ws("_", col("city"), col("poiID").cast("string")))
)

w_pop = Window.partitionBy("city").orderBy(F.desc("actual_visits"))

poi_popularity = (
    poi_popularity
    .withColumn("popularity_rank", F.row_number().over(w_pop))
)

In [14]:
print('Top 1 POI Popularity in each city:')
poi_popularity.select("city", "poi_key", "actual_visits").where(col('popularity_rank') == 1).show(truncate=False)

Top 1 POI Popularity in each city:


26/06/02 16:07:19 WARN TaskSetManager: Stage 1134 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.


+---------+------------+-------------+
|city     |poi_key     |actual_visits|
+---------+------------+-------------+
|Athens   |Athens_4    |930          |
|Barcelona|Barcelona_29|1593         |
|Budapest |Budapest_8  |1584         |
|Edinburgh|Edinburgh_1 |4569         |
|Glasgow  |Glasgow_1   |1491         |
|London   |London_5    |10610        |
|Madrid   |Madrid_2    |9028         |
|Melbourne|Melbourne_71|1693         |
|NewDelhi |NewDelhi_12 |709          |
|Osaka    |Osaka_8     |1307         |
|Perth    |Perth_6     |500          |
|Toronto  |Toronto_11  |4139         |
|Vienna   |Vienna_17   |5527         |
+---------+------------+-------------+



## Compare PageRank with POI popularity
A POI can be highly popular but not central in the graph. Conversely, a POI can have high PageRank because it connects well to other important POIs.

In [15]:
comparison_df = (
    pagerank_df.join(poi_popularity.select("poi_key", "actual_visits", "popularity_rank"),
                     pagerank_df["id"] == poi_popularity["poi_key"], "left")
                .fillna({"actual_visits": 0})
)

comparison_df = (
    comparison_df
    .withColumn(
        "rank_difference",
        F.abs(col("popularity_rank") - col("pagerank_rank"))
    )
    .withColumn(
        "Judge",
        F.when(col("popularity_rank") - col("pagerank_rank") > 0, 1)
        .when(col("popularity_rank") - col("pagerank_rank") < 0, -1)
        .otherwise(0)
    )
)

comparison_df.show(100, truncate=False)

26/06/02 16:07:30 WARN TaskSetManager: Stage 1197 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.


+------------+---------+--------------------------------------------------------+----------+----------+----------+-------------------+-------------+------------+-------------+---------------+---------------+-----+
|id          |city     |poiName                                                 |category  |lat       |lon       |pagerank           |pagerank_rank|poi_key     |actual_visits|popularity_rank|rank_difference|Judge|
+------------+---------+--------------------------------------------------------+----------+----------+----------+-------------------+-------------+------------+-------------+---------------+---------------+-----+
|Athens_1    |Athens   |Acropolis_of_Athens                                     |Historical|37.9715323|23.7257492|1.0904063221273281 |1            |Athens_1    |836          |2              |1              |1    |
|Athens_2    |Athens   |Acropolis_Museum                                        |Museum    |37.9684499|23.7285227|1.0904063221273281 |2         

The `Judge` column compares popularity rank against PageRank rank.

```Judge > 0``` : The POI has better PageRank than popularity. It may be structurally important even if tourists do not visit it as frequently.

```Judge < 0``` : The POI is more popular than its graph importance. It attracts many tourists but may not be central in the travel network.

```Judge = 0``` : The POI is both popular and structurally important.

## Save comparison table

In [16]:
raw_data_path = project_root / "Datasets" / "Raw"
analysis_path = project_root / "Analysis"
figure_dir = analysis_path / "figures"
output_dir = analysis_path / "outputs"
comparison_output = output_dir / "pagerank_vs_popularity.csv"
output_dir.mkdir(parents=True, exist_ok=True)
comparison_df.select("city", "poiName", "category", "lat", "lon", 
                     "pagerank_rank", "popularity_rank", "rank_difference", "Judge"
                     ).toPandas().to_csv(comparison_output, index=False)

print("Saved comparison table to:", comparison_output)

26/06/02 16:07:41 WARN TaskSetManager: Stage 1325 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.


Saved comparison table to: /home/duongthai/Downloads/DoAnBigData/smart-tourism/Analysis/outputs/pagerank_vs_popularity.csv


## Calculate Avarage Duration `dur(v_i)`

In [17]:
dur_df = (
    visit_df
    .groupBy("city", "poiID", "userID")
    .agg(
        F.min("dateTaken").alias("t_arrival"),
        F.max("dateTaken").alias("t_depart"),
    )
    .withColumn("visit_duration_sec", col("t_depart") - col("t_arrival"))
    .filter(col("visit_duration_sec") > 0)   # loại bỏ single-photo visits
)


In [18]:
avg_dur_df = (
    dur_df
    .groupBy("city", "poiID")
    .agg(
        F.count("*").alias("pop_count"),
        F.mean("visit_duration_sec").alias("dur_avg_sec"),
        F.expr("percentile(visit_duration_sec, 0.75)").alias("dur_p75_sec"),
    )
    .withColumn("dur_avg_min", F.round(col("dur_avg_sec") / 60, 2))
    .withColumn("dur_p75_min", F.round(col("dur_p75_sec") / 60, 2))
)

avg_dur_df.join(poi_df.select("city", "poiID", "poiName"), ["city", "poiID"]
                ).orderBy("city", col("dur_avg_min").desc()
                ).show(15, truncate=False)



26/06/02 16:07:53 WARN TaskSetManager: Stage 1397 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.


+------+-----+---------+------------------+-----------+-----------+-----------+--------------------------------------------------------+
|city  |poiID|pop_count|dur_avg_sec       |dur_p75_sec|dur_avg_min|dur_p75_min|poiName                                                 |
+------+-----+---------+------------------+-----------+-----------+-----------+--------------------------------------------------------+
|Athens|16   |9        |6550096.777777778 |9913.0     |109168.28  |165.22     |Ethniko_Mouseio_Sugkhrones_Tekhnes_(EMST)               |
|Athens|25   |13       |4219032.615384615 |2445.0     |70317.21   |40.75      |Museums_of_the_City_of_Athens_Vouros_Eutaxias_Foundation|
|Athens|4    |33       |3828468.0606060605|11240.0    |63807.8    |187.33     |National_Archaeological_Museum                          |
|Athens|27   |8        |3616631.75        |3848895.75 |60277.2    |64148.26   |Municipal_Gallery_of_Athens                             |
|Athens|19   |38       |2671675.078947368

In [19]:
avg_dur_df.select("dur_avg_min").summary("count","min","25%","50%","75%","max").show()

26/06/02 16:08:02 WARN TaskSetManager: Stage 1405 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.


+-------+-----------+
|summary|dur_avg_min|
+-------+-----------+
|  count|        411|
|    min|       1.88|
|    25%|   29876.12|
|    50%|  123311.01|
|    75%|  310072.33|
|    max| 7062463.63|
+-------+-----------+



## Calculate user interest by category

In [20]:
# Merge visit + duration + POI category
user_poi_dur = (
    dur_df
    .join(avg_dur_df.select("city", "poiID", "dur_avg_sec"),
          ["city", "poiID"], "left")
    .join(poi_df.select("city", "poiID", "poiTheme"),
          ["city", "poiID"], "left")
    # int_u(c) = sum( visit_duration / dur(poi) ) cho các POI thuộc category c
    .withColumn("interest_contribution",
                col("visit_duration_sec") / col("dur_avg_sec"))
)


In [21]:
user_interest_df = (
    user_poi_dur
    .groupBy("userID", "city", "poiTheme")
    .agg(F.sum("interest_contribution").alias("int_u_c"))
    .orderBy("userID", "city", col("int_u_c").desc())
)

user_interest_df.show(20, truncate=False)



26/06/02 16:08:09 WARN TaskSetManager: Stage 1415 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.
26/06/02 16:08:12 WARN TaskSetManager: Stage 1417 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.


+-------------+---------+-------------+---------------------+
|userID       |city     |poiTheme     |int_u_c              |
+-------------+---------+-------------+---------------------+
|10002536@N02 |Vienna   |Structure    |5.119431341158671E-5 |
|10006319@N00 |Madrid   |Museum       |0.03398049141434036  |
|10006319@N00 |Madrid   |Historical   |1.8467716725561028E-4|
|10006319@N00 |Madrid   |Transport    |1.2286076495017584E-5|
|100108211@N06|Barcelona|Museum       |0.18203054606921065  |
|10012675@N05 |Toronto  |Cultural     |0.00898083740249038  |
|10014440@N06 |Toronto  |Shopping     |3.5616631127283083   |
|10014440@N06 |Toronto  |Amusement    |5.45619131754783E-4  |
|10014440@N06 |Vienna   |Palace       |0.02606126063613024  |
|10014440@N06 |Vienna   |Museum       |1.2118426840719367E-4|
|100153087@N08|London   |Museum       |0.0025088025618991484|
|100183713@N05|Madrid   |Museum       |2.8045423462295083E-6|
|100228151@N03|London   |Transport    |3.1388356786612195E-4|
|1002439

In [22]:
# Save for RAG
user_interest_df.write.mode("overwrite").parquet(str(project_root / "Datasets" / "GraphFrames" / "user_interest"))
avg_dur_df.write.mode("overwrite").parquet(str(project_root / "Datasets" / "GraphFrames" / "poi_duration"))

26/06/02 16:08:24 WARN TaskSetManager: Stage 1435 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.
26/06/02 16:08:25 WARN TaskSetManager: Stage 1436 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.
26/06/02 16:08:39 WARN TaskSetManager: Stage 1462 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.


## Calculate `Profit_u(v_i)`, `Cost_u(v_i, v_j)` (Eq. 4, 5)

In [23]:
ALPHA = 0.5  # trọng số cân bằng user interest vs POI popularity (theo bài báo)

# Normalize PageRank về [0,1] theo từng city
city_window = Window.partitionBy("city")

pr_norm = (
    pagerank_df
    .withColumn("pr_min", F.min("pagerank").over(city_window))
    .withColumn("pr_max", F.max("pagerank").over(city_window))
    .withColumn(
        "pop_norm",
        (F.col("pagerank") - F.col("pr_min")) /
        (F.col("pr_max") - F.col("pr_min") + 1e-9)
    )
    .select("id", "city", "poiName", "category", "lat", "lon", "pop_norm")
)


In [24]:
# Normalize int_u(c) về [0,1] theo từng (user, city)
user_city_window = Window.partitionBy("userID", "city")
int_norm = (
    user_interest_df
    .withColumn("int_min", F.min("int_u_c").over(user_city_window))
    .withColumn("int_max", F.max("int_u_c").over(user_city_window))
    .withColumn(
        "int_norm",
        (col("int_u_c") - col("int_min")) /
        (col("int_max") - col("int_min") + 1e-9)
    )
    .select("userID", "city", "poiTheme", "int_norm")
)


In [25]:
# Profit_u(v_i) = alpha * int_norm + (1-alpha) * pop_norm
profit_df = (
    pr_norm.join(int_norm,(int_norm["city"] == pr_norm["city"]) & (int_norm["poiTheme"] == pr_norm["category"]), "left"
        ).withColumn("profit", ALPHA * F.coalesce(col("int_norm"), F.lit(0)) + (1 - ALPHA) * col("pop_norm")
        ).select("userID", pr_norm["city"], "id", "poiName", "category", "profit", "pop_norm",
            F.coalesce(col("int_norm"), F.lit(0)).alias("int_norm")
        ).filter( (col("profit") > 0) &  (col("userID").isNotNull())
        )
)

In [26]:
print("Profit_u(v_i) - Exp:")
profit_df.orderBy("userID", col("profit").desc()).show(15, truncate=False)

profit_df.write.mode("overwrite").parquet(str(project_root / "Datasets" / "GraphFrames" / "poi_profit"))

Profit_u(v_i) - Exp:


26/06/02 16:08:58 WARN TaskSetManager: Stage 1527 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.
26/06/02 16:08:58 WARN TaskSetManager: Stage 1528 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.


+------------+------+---------+---------------------------------------------+---------+-------------------+------------------+------------------+
|userID      |city  |id       |poiName                                      |category |profit             |pop_norm          |int_norm          |
+------------+------+---------+---------------------------------------------+---------+-------------------+------------------+------------------+
|10002536@N02|Vienna|Vienna_8 |Donauturm                                    |Structure|0.49999999945497214|0.9999999989099443|0.0               |
|10006319@N00|Madrid|Madrid_30|Museum_of_the_Americas                       |Museum   |0.4999999852803532 |0.0               |0.9999999705607064|
|10006319@N00|Madrid|Madrid_29|Museum_ABC                                   |Museum   |0.4999999852803532 |0.0               |0.9999999705607064|
|10006319@N00|Madrid|Madrid_28|Real_Academia_de_Bellas_Artes_de_San_Fernando|Museum   |0.4999999852803532 |0.0              

26/06/02 16:09:13 WARN TaskSetManager: Stage 1681 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.
26/06/02 16:09:14 WARN TaskSetManager: Stage 1682 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.


## BFS For All Cities

In [27]:
# ── Tham số toàn cục ───────────────────────────────────────────────────────
BFS_BUDGET_MIN = 480   # 8 giờ
BFS_MAX_POIS   = 5     # độ sâu tối đa (tăng lên 6-7 nếu RAM cho phép)
BFS_TOP_K      = 3     # số itinerary tốt nhất lưu lại mỗi user

profit_pdf_all  = profit_df.toPandas()
avg_dur_pdf_all = avg_dur_df.toPandas()
# dist_pdf đã là pandas từ cell load dữ liệu

all_cities = sorted(profit_pdf_all["city"].dropna().unique().tolist())
print(f"Tổng số thành phố: {len(all_cities)}")
print(f"Cities: {all_cities}")

26/06/02 16:09:31 WARN TaskSetManager: Stage 1831 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.
26/06/02 16:09:32 WARN TaskSetManager: Stage 1832 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.
26/06/02 16:09:52 WARN TaskSetManager: Stage 1922 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.


Tổng số thành phố: 13
Cities: ['Athens', 'Barcelona', 'Budapest', 'Edinburgh', 'Glasgow', 'London', 'Madrid', 'Melbourne', 'NewDelhi', 'Osaka', 'Perth', 'Toronto', 'Vienna']


In [28]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    ArrayType, IntegerType
)
import math

def find_itineraries_bfs(
    spark, g, city: str, user_id: str,
    profit_pdf,        # pandas DataFrame: [id, poiName, category, profit]
    dur_pdf,            # pandas DataFrame: [poiID, dur_avg_min]
    dist_pdf_city,     # pandas DataFrame: [fromPOIid, toPOIid, duration]
    budget_min: float = 480,   # 8 giờ = 480 phút
    max_pois: int = 5,         # giới hạn độ sâu tìm kiếm
    top_k: int = 3,            # số itinerary tốt nhất trả về
):
    """
    BFS tìm kiếm itinerary tốt nhất trong ngân sách thời gian.
    Trả về danh sách (path, total_profit, total_time_min).
    """
    # Index profit và duration
    profit_map = dict(zip(profit_pdf["id"], profit_pdf["profit"]))
    dur_map    = {str(int(r["poiID"])): r["dur_avg_min"] for _, r in dur_pdf.iterrows()}
    # Travel time matrix (phút)
    travel_map = {}
    for _, row in dist_pdf_city.iterrows():
        travel_map[(str(int(row["fromPOIid"])), str(int(row["toPOIid"])))] = \
            row["duration"] / 60  # giây -> phút

    all_ids = list(profit_map.keys())

    # BFS state: (current_poi_id, path_list, total_profit, total_time)
    from collections import deque
    queue   = deque()
    results = []

    # Khởi tạo: mỗi POI là điểm xuất phát
    for start_id in all_ids:
        poi_num = start_id.split("_", 1)[-1]
        dur = dur_map.get(poi_num, 30)
        if dur <= budget_min:
            queue.append((start_id, [start_id], profit_map.get(start_id, 0), dur))

    visited_paths = set()

    while queue:
        cur_id, path, total_profit, total_time = queue.popleft()

        # Lưu kết quả (itinerary có ít nhất 1 POI)
        path_key = tuple(path)
        if path_key not in visited_paths:
            visited_paths.add(path_key)
            results.append((list(path), total_profit, total_time))

        # Mở rộng nếu chưa đạt giới hạn độ sâu
        if len(path) >= max_pois:
            continue

        cur_num = cur_id.split("_", 1)[-1]

        for next_id in all_ids:
            if next_id in path:
                continue
            next_num = next_id.split("_", 1)[-1]
            travel   = travel_map.get((cur_num, next_num), 999)
            stay     = dur_map.get(next_num, 30)
            new_time = total_time + travel + stay
            if new_time <= budget_min:
                queue.append((next_id, path + [next_id],
                              total_profit + profit_map.get(next_id, 0),
                              new_time))

    # Sắp xếp theo profit giảm dần, trả top_k
    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_k]

In [29]:
import time
all_bfs_rows = []
total_start  = time.time()

for city in all_cities:
    city_start = time.time()

    profit_city = profit_pdf_all[profit_pdf_all["city"] == city].copy()
    dur_city    = avg_dur_pdf_all[avg_dur_pdf_all["city"] == city].copy()
    dist_city   = distance_pdf[distance_pdf["city"] == city].copy()

    # Tra tên POI nhanh bằng dict
    id_to_name  = dict(zip(profit_city["id"], profit_city["poiName"]))

    all_users = profit_city["userID"].dropna().unique().tolist()
    n_users   = len(all_users)

    print(f"\n{'─'*60}")
    print(f"[{city}]  {n_users} users | POIs: {dist_city['fromPOIid'].nunique()}")

    city_rows = []
    for u_idx, user_id in enumerate(all_users):
        user_profit = profit_city[profit_city["userID"] == user_id]
        if user_profit.empty:
            continue

        itineraries = find_itineraries_bfs(
            spark=spark, g=g,
            city=city, user_id=user_id,
            profit_pdf=user_profit,
            dur_pdf=dur_city,
            dist_pdf_city = dist_city,
            budget_min=BFS_BUDGET_MIN,
            max_pois=BFS_MAX_POIS,
            top_k=BFS_TOP_K,
        )

        for rank, (path, profit, time_min) in enumerate(itineraries, 1):
            names = [id_to_name.get(p, p).replace("_", " ") for p in path]
            import builtins

            city_rows.append({
                "city": city,
                "userID": user_id,
                "rank": rank,
                "path_ids": "|".join(path),
                "path_names": " → ".join(names),
                "n_pois": len(path),
                "total_profit": builtins.round(float(profit), 4),
                "total_time_min": builtins.round(float(time_min), 1),
            })  

        # Progress mỗi 50 user
        if (u_idx + 1) % 50 == 0 or (u_idx + 1) == n_users:
            print(f"  {u_idx+1:>4}/{n_users} users done "
                  f"({time.time()-city_start:.0f}s elapsed)")

    all_bfs_rows.extend(city_rows)
    print(f"  ✓ {len(city_rows)} itinerary rows — "
          f"{time.time()-city_start:.1f}s")



────────────────────────────────────────────────────────────
[Athens]  291 users | POIs: 24
    50/291 users done (2s elapsed)
   100/291 users done (4s elapsed)
   150/291 users done (5s elapsed)
   200/291 users done (7s elapsed)
   250/291 users done (9s elapsed)
   291/291 users done (11s elapsed)
  ✓ 863 itinerary rows — 10.6s

────────────────────────────────────────────────────────────
[Barcelona]  782 users | POIs: 29
    50/782 users done (1s elapsed)
   100/782 users done (2s elapsed)
   150/782 users done (4s elapsed)
   200/782 users done (5s elapsed)
   250/782 users done (7s elapsed)
   300/782 users done (9s elapsed)
   350/782 users done (10s elapsed)
   400/782 users done (12s elapsed)
   450/782 users done (13s elapsed)
   500/782 users done (15s elapsed)
   550/782 users done (17s elapsed)
   600/782 users done (19s elapsed)
   650/782 users done (22s elapsed)
   700/782 users done (24s elapsed)
   750/782 users done (25s elapsed)
   782/782 users done (26s elapsed)

In [30]:
# ── Tổng hợp và lưu ───────────────────────────────────────────────────────
bfs_df = pd.DataFrame(all_bfs_rows)
print(f"\n{'='*60}")
print(f"Tổng rows: {len(bfs_df):,} | {len(all_cities)} cities | "
      f"{bfs_df['userID'].nunique():,} unique users")
print(f"Wall time: {time.time()-total_start:.1f}s")

# Lưu Parquet để dùng cho RAG
bfs_spark_df = spark.createDataFrame(bfs_df)
bfs_spark_df.write.mode("overwrite").partitionBy("city") \
    .parquet(str(project_root / "Datasets" / "GraphFrames" / "bfs_itineraries"))

# Lưu CSV để xem nhanh
bfs_df.to_csv(project_root/"Datasets"/"GraphFrames"/"bfs_itineraries_all.csv", index=False)
print("Đã lưu bfs_itineraries/ (Parquet, partitioned by city)")
print("Đã lưu bfs_itineraries_all.csv")



Tổng rows: 5,485 | 13 cities | 2,779 unique users
Wall time: 289.6s


Đã lưu bfs_itineraries/ (Parquet, partitioned by city)
Đã lưu bfs_itineraries_all.csv


In [31]:
# Thống kê tóm tắt theo city
summary = (
    bfs_df[bfs_df["rank"] == 1]   # chỉ itinerary tốt nhất mỗi user
    .groupby("city")
    .agg(
        n_users       = ("userID",        "nunique"),
        avg_profit    = ("total_profit",  "mean"),
        avg_pois      = ("n_pois",        "mean"),
        avg_time_min  = ("total_time_min","mean"),
    )
    .round(3)
)
print("\nThống kê BFS (itinerary rank-1) theo city:")
print(summary.to_string())



Thống kê BFS (itinerary rank-1) theo city:
           n_users  avg_profit  avg_pois  avg_time_min
city                                                  
Athens         291       2.304     3.821       165.669
Barcelona      575       0.640     1.054        70.345
Budapest       295       0.410     1.000        24.237
Edinburgh      436       0.452     1.000        30.000
Glasgow        213       0.689     1.113       222.082
London          34       0.714     2.000       475.900
Madrid         134       0.776     1.813       174.623
Melbourne      324       1.601     2.380       179.688
NewDelhi       139       1.465     2.173       214.799
Osaka           72       0.454     1.000        20.750
Perth           19       0.450     1.000        19.800
Toronto        314       0.592     1.000        47.506
Vienna         158       0.297     1.000        30.000


## Motif-finding

In [32]:
t0 = time.time()

motifs_raw = g.find("(a)-[e1]->(b); (b)-[e2]->(c)")

motifs_all = (
    motifs_raw
    # Chỉ lấy motif cùng city
    .filter(F.col("a.city") == F.col("b.city"))
    .filter(F.col("b.city") == F.col("c.city"))
    # Không trùng POI trong bộ 3
    .filter(F.col("a.id") != F.col("b.id"))
    .filter(F.col("b.id") != F.col("c.id"))
    .filter(F.col("a.id") != F.col("c.id"))
    .select(
        F.col("a.city").alias("city"),
        F.col("a.id").alias("id_A"),
        F.col("b.id").alias("id_B"),
        F.col("c.id").alias("id_C"),
        F.col("a.poiName").alias("poi_A"),
        F.col("b.poiName").alias("poi_B"),
        F.col("c.poiName").alias("poi_C"),
        F.col("a.category").alias("cat_A"),
        F.col("b.category").alias("cat_B"),
        F.col("c.category").alias("cat_C"),
        F.col("e1.duration").alias("travel_AB_sec"),
        F.col("e2.duration").alias("travel_BC_sec"),
    )
    .withColumn(
        "total_travel_min",
        F.round((F.col("travel_AB_sec") + F.col("travel_BC_sec")) / 60, 2)
    )
    .filter(F.col("total_travel_min") <= 30)
    .orderBy("city", "total_travel_min")
    .cache()   # cache vì sẽ dùng nhiều lần bên dưới
)

/home/duongthai/.local/lib/python3.10/site-packages/pyspark/sql/classic/dataframe.py:128: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


In [33]:
total_motifs = motifs_all.count()
print(f"✓ Tổng số motif (A→B→C, travel ≤ 30 phút): {total_motifs:,}")
print(f"  Wall time: {time.time()-t0:.1f}s")

# Thống kê số motif theo city
print("\nSố motif theo city:")
motif_city_count = (
    motifs_all
    .groupBy("city")
    .count()
    .withColumnRenamed("count", "n_motifs")
    .orderBy(F.col("n_motifs").desc())
)
motif_city_count.show(20, truncate=False)

# Top-5 motif ngắn nhất mỗi city
print("Top-5 motif đường đi ngắn nhất mỗi city:")
from pyspark.sql import Window

city_rank_window = Window.partitionBy("city").orderBy("total_travel_min")

top_motifs_per_city = (
    motifs_all
    .withColumn("city_rank", F.row_number().over(city_rank_window))
    .filter(F.col("city_rank") <= 5)
    .select("city", "city_rank", "poi_A", "poi_B", "poi_C",
            "cat_A", "cat_B", "cat_C", "total_travel_min")
    .orderBy("city", "city_rank")
)
top_motifs_per_city.show(65, truncate=False)   # 13 cities × 5

# Phân tích cặp category hay đi cùng nhau (A→B) 
print("Top cặp category (A → B) hay xuất hiện trong motif:")
category_pair_freq = (
    motifs_all
    .groupBy("city", "cat_A", "cat_B")
    .count()
    .withColumnRenamed("count", "freq")
    .orderBy("city", F.col("freq").desc())
)
category_pair_freq.show(40, truncate=False)


✓ Tổng số motif (A→B→C, travel ≤ 30 phút): 301,258
  Wall time: 40.6s

Số motif theo city:


+---------+--------+
|city     |n_motifs|
+---------+--------+
|Melbourne|230462  |
|Edinburgh|16584   |
|Vienna   |13316   |
|Athens   |11168   |
|Budapest |6950    |
|Perth    |6646    |
|Madrid   |5832    |
|Glasgow  |4584    |
|Toronto  |2532    |
|Barcelona|2104    |
|London   |838     |
|Osaka    |226     |
|NewDelhi |16      |
+---------+--------+

Top-5 motif đường đi ngắn nhất mỗi city:


+---------+---------+--------------------------------------------------------+---------------------------+--------------------------------------------------------+-------------+-------------+-------------+----------------+
|city     |city_rank|poi_A                                                   |poi_B                      |poi_C                                                   |cat_A        |cat_B        |cat_C        |total_travel_min|
+---------+---------+--------------------------------------------------------+---------------------------+--------------------------------------------------------+-------------+-------------+-------------+----------------+
|Athens   |1        |Old_Parliament                                          |National_Historical_Museum |Museums_of_the_City_of_Athens_Vouros_Eutaxias_Foundation|Museum       |Museum       |Museum       |3.42            |
|Athens   |2        |Museums_of_the_City_of_Athens_Vouros_Eutaxias_Foundation|National_Historical_Museum |Ol

+---------+----------+----------+----+
|city     |cat_A     |cat_B     |freq|
+---------+----------+----------+----+
|Athens   |Museum    |Museum    |6284|
|Athens   |Museum    |Historical|1811|
|Athens   |Historical|Museum    |1785|
|Athens   |Education |Museum    |316 |
|Athens   |Historical|Historical|308 |
|Athens   |Museum    |Education |208 |
|Athens   |Park      |Museum    |137 |
|Athens   |Education |Historical|123 |
|Athens   |Historical|Education |98  |
|Athens   |Museum    |Park      |96  |
|Athens   |Historical|Park      |2   |
|Barcelona|Museum    |Museum    |1204|
|Barcelona|Shopping  |Museum    |310 |
|Barcelona|Museum    |Shopping  |231 |
|Barcelona|Zoo       |Museum    |98  |
|Barcelona|Education |Museum    |62  |
|Barcelona|Museum    |Zoo       |42  |
|Barcelona|Zoo       |Shopping  |29  |
|Barcelona|Shopping  |Zoo       |26  |
|Barcelona|Education |Shopping  |25  |
|Barcelona|Historical|Museum    |12  |
|Barcelona|Museum    |Historical|12  |
|Barcelona|Museum    |Edu

In [34]:
# Lưu kết quả motif
motifs_all.write.mode("overwrite").partitionBy("city") \
    .parquet(str(project_root / "Datasets" / "GraphFrames" / "motifs_all"))

top_motifs_per_city.write.mode("overwrite") \
    .parquet(str(project_root / "Datasets" / "GraphFrames" / "motifs_top5_per_city"))

category_pair_freq.write.mode("overwrite") \
    .parquet(str(project_root / "Datasets" / "GraphFrames"/ "motif_category_pairs"))

# Lưu CSV tóm tắt để xem nhanh
motif_city_count.toPandas().to_csv(
    project_root / "Datasets" / "GraphFrames" / "motif_city_summary.csv", index=False)
top_motifs_per_city.toPandas().to_csv(
    project_root / "Datasets" / "GraphFrames" / "motif_top5_per_city.csv", index=False)
category_pair_freq.toPandas().to_csv(
    project_root / "Datasets" / "GraphFrames" / "motif_category_pairs.csv", index=False)

print(f"Đã lưu motif outputs:")

#motifs_all.unpersist()

Đã lưu motif outputs:


In [35]:
# Lưu đồ thị và các kết quả tính toán
vertices.write.mode("overwrite").parquet(str(project_root / "Datasets" / "GraphFrames" / "vertices"))
edges_full.write.mode("overwrite").parquet(str(project_root / "Datasets" / "GraphFrames" / "edges"))
pagerank_df.write.mode("overwrite").parquet(str(project_root / "Datasets" / "GraphFrames" / "pagerank"))
comparison_df.write.mode("overwrite").parquet(str(project_root / "Datasets" / "GraphFrames" / "pagerank_vs_actual"))

26/06/02 16:17:14 WARN TaskSetManager: Stage 2240 contains a task of very large size (3624 KiB). The maximum recommended task size is 1000 KiB.
